In [ ]:
%%configure -f
{
    "conf": {
        "spark.dynamicAllocation.enabled": "false",
        "spark.driver.cores": "4",
        "spark.driver.memory": "28g",
        "spark.executor.cores": "4",
        "spark.executor.memory": "28g",
        "spark.executor.instances": 1
    }
}

# Clustering

**Scenario:** supermarket_net_sales_forecast  
**Generated:** 2026-06-03  
**Key Parameters:**
- Algorithm: K-Means on normalized weekly sales curves (shape-based)
- Clusters: k=4 for regular stores + 1 erratic group
- Normalization: Z-score per store (removes magnitude, keeps pattern)
- Silhouette: 0.053 (weak but best balanced option)

This notebook clusters the 38 regular stores into 4 groups based on the **shape** of their normalized weekly sales curves using K-Means. The 12 erratic stores are assigned to their own group.

## Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from kneed import KneeLocator

print("Packages loaded.")

## Setup & Configuration

In [ ]:
# CUSTOMIZED: Column and table names for supermarket scenario
date_var = 'WEEK_START_DT'
unique_id = 'STORE_LOCATION_ID'
y = 'TOTAL_NET_SALES'

# Fabric Lakehouse configuration
TABLE_PREFIX = "`ts-forecaster-ws1`.ts_mmm.dbo"
INPUT_TABLE = f"{TABLE_PREFIX}.supermarket_net_sales_forecast_profiled"
OUTPUT_TABLE = f"{TABLE_PREFIX}.supermarket_net_sales_forecast_clustered"

# CUSTOMIZED: K-Means parameters
chosen_k = 4  # Determined via elbow + silhouette analysis
kmeans_kwargs = {
    "init": "k-means++",
    "n_init": 10,
    "max_iter": 300,
    "random_state": 42,
}
max_k_to_try = 10  # For elbow/silhouette exploration

print("Configuration loaded:")
print(f"  - Input table: {INPUT_TABLE}")
print(f"  - Output table: {OUTPUT_TABLE}")
print(f"  - Chosen clusters: {chosen_k}")

## Load Data

In [ ]:
# Load profiled data from Lakehouse
df_spark = spark.table(INPUT_TABLE)
df = df_spark.toPandas()
print(f"Loaded profiled data: {df.shape}")
print(f"Profile distribution:")
print(df[[unique_id, 'profile']].drop_duplicates()['profile'].value_counts())
print(f"\nStore archetypes:")
print(df[[unique_id, 'STORE_ARCHETYPE']].drop_duplicates()['STORE_ARCHETYPE'].value_counts())

## Filter Regular Stores & Normalize Sales Curves

K-Means is applied only to **regular** stores. Each store's 104-week sales curve is z-score normalized to remove volume differences and focus on the **shape** of the pattern (seasonality, trends).

In [ ]:
# Filter to regular stores
regular_stores = df.loc[df['profile'] == 'regular', unique_id].unique()
print(f"Regular stores for clustering: {len(regular_stores)}")

# Pivot: rows=stores, columns=weeks, values=TOTAL_NET_SALES
df_regular = df[df[unique_id].isin(regular_stores)].copy()
df_pivot = df_regular.pivot(index=unique_id, columns=date_var, values=y)
print(f"Pivot shape (stores x weeks): {df_pivot.shape}")
print(f"Any NaN: {df_pivot.isna().any().any()}")

# CUSTOMIZED: Z-score normalize per store (shape-based clustering)
scaler = StandardScaler()
df_normalized = pd.DataFrame(
    scaler.fit_transform(df_pivot.T).T,
    index=df_pivot.index,
    columns=df_pivot.columns
)
print(f"\nNormalized shape: {df_normalized.shape}")
print(f"Verification - Store mean: {df_normalized.iloc[0].mean():.6f}, std: {df_normalized.iloc[0].std():.4f}")

## Elbow Method & Silhouette Analysis

### âœ… CHECK POINT: Determine optimal number of clusters

In [ ]:
X = df_normalized.values  # (38 stores, 104 weeks)

# Elbow method - SSE for k=1..max_k
sse = []
for k in range(1, max_k_to_try + 1):
    km = KMeans(n_clusters=k, **kmeans_kwargs)
    km.fit(X)
    sse.append(km.inertia_)

# Find elbow
kl = KneeLocator(range(1, max_k_to_try + 1), sse, curve="convex", direction="decreasing")
elbow_k = kl.elbow
print(f"Elbow method optimal k: {elbow_k}")

# Plot elbow
plt.figure(figsize=(10, 5))
plt.plot(range(1, max_k_to_try + 1), sse, marker='o')
if elbow_k:
    plt.axvline(x=elbow_k, color='r', linestyle='--', label=f'Elbow at k={elbow_k}')
plt.xlabel('Number of Clusters')
plt.ylabel('SSE (Inertia)')
plt.title('Elbow Method')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Silhouette coefficients for k=2..max_k
silhouette_scores = []
for k in range(2, max_k_to_try + 1):
    km = KMeans(n_clusters=k, **kmeans_kwargs)
    labels = km.fit_predict(X)
    score = silhouette_score(X, labels)
    silhouette_scores.append((k, score))

print("Silhouette scores:")
for k, s in silhouette_scores:
    marker = " â† best" if s == max(sc for _, sc in silhouette_scores) else ""
    print(f"  k={k}: {s:.4f}{marker}")

best_sil_k = max(silhouette_scores, key=lambda x: x[1])[0]
print(f"\nBest silhouette k: {best_sil_k}")

# Plot silhouette
plt.figure(figsize=(10, 5))
plt.plot([k for k, _ in silhouette_scores], [s for _, s in silhouette_scores], marker='o')
plt.axvline(x=best_sil_k, color='r', linestyle='--', label=f'Best at k={best_sil_k}')
plt.xlabel('Number of Clusters')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Method')
plt.legend()
plt.tight_layout()
plt.show()

## Execute K-Means (k=4)

### âœ… CHECK POINT: Review cluster assignments

In [ ]:
# CUSTOMIZED: Final K-Means with k=4 (chosen based on analysis above)
km_final = KMeans(n_clusters=chosen_k, **kmeans_kwargs)
labels_final = km_final.fit_predict(X)

# Build cluster assignment for regular stores
store_meta = df[[unique_id, 'STORE_ARCHETYPE']].drop_duplicates().set_index(unique_id)
df_cluster_assign = pd.DataFrame({'cluster': labels_final}, index=df_normalized.index)
df_cluster_assign = df_cluster_assign.join(store_meta)

# Add erratic stores
erratic_stores = df.loc[df['profile'] == 'erratic', unique_id].unique()
erratic_df = pd.DataFrame({unique_id: erratic_stores}).set_index(unique_id).join(store_meta)
erratic_df['cluster'] = -1

# Combine all stores
all_stores_clustered = pd.concat([df_cluster_assign, erratic_df])
all_stores_clustered['profile_cluster'] = all_stores_clustered['cluster'].apply(
    lambda x: f'regular_{x}' if x >= 0 else 'erratic'
)

print(f"All stores clustered: {len(all_stores_clustered)}")
print(f"\nProfile-cluster distribution:")
print(all_stores_clustered['profile_cluster'].value_counts().sort_index())
print(f"\nCrosstab profile_cluster vs STORE_ARCHETYPE:")
print(pd.crosstab(all_stores_clustered['profile_cluster'], all_stores_clustered['STORE_ARCHETYPE'], margins=True))

## Cluster Visualization

In [ ]:
# Plot cluster centroids (normalized patterns)
dates = pd.to_datetime(df_normalized.columns)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('K-Means Cluster Centroids (Normalized Sales Patterns, k=4)', fontsize=14)

for ax, cluster_id in zip(axes.flat, range(chosen_k)):
    stores_in_cluster = df_cluster_assign[df_cluster_assign['cluster'] == cluster_id].index
    cluster_data = df_normalized.loc[stores_in_cluster]
    
    for store_id in stores_in_cluster:
        ax.plot(dates, cluster_data.loc[store_id].values, alpha=0.3, linewidth=0.8)
    
    centroid = cluster_data.mean(axis=0).values
    ax.plot(dates, centroid, 'k-', linewidth=2.5, label='Centroid')
    
    archetypes = df_cluster_assign.loc[stores_in_cluster, 'STORE_ARCHETYPE'].value_counts()
    arch_str = ', '.join([f"{k[0]}:{v}" for k, v in archetypes.items()])
    ax.set_title(f'Cluster {cluster_id} ({len(stores_in_cluster)} stores: {arch_str})')
    ax.set_ylabel('Normalized Sales')
    ax.legend(loc='upper left')
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Plot raw sales per cluster
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('K-Means Clusters: Raw Weekly Sales (k=4)', fontsize=14)

for ax, cluster_id in zip(axes.flat, range(chosen_k)):
    stores_in_cluster = df_cluster_assign[df_cluster_assign['cluster'] == cluster_id].index
    cluster_raw = df_pivot.loc[stores_in_cluster]
    
    for store_id in stores_in_cluster:
        ax.plot(dates, cluster_raw.loc[store_id].values, alpha=0.4, linewidth=0.8)
    
    centroid_raw = cluster_raw.mean(axis=0).values
    ax.plot(dates, centroid_raw, 'k-', linewidth=2.5, label=f'Mean: ${centroid_raw.mean():,.0f}/wk')
    
    archetypes = df_cluster_assign.loc[stores_in_cluster, 'STORE_ARCHETYPE'].value_counts()
    arch_str = ', '.join([f"{k[0]}:{v}" for k, v in archetypes.items()])
    ax.set_title(f'Cluster {cluster_id} ({len(stores_in_cluster)} stores: {arch_str})')
    ax.set_ylabel('Net Sales ($)')
    ax.legend(loc='upper left')
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    ax.tick_params(axis='x', rotation=45)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.show()

## Merge & Save to Lakehouse

Merge `profile_cluster` into the full 5,200-row dataset and save as the clustered table.

In [ ]:
# Merge profile_cluster into full dataset
cluster_map = all_stores_clustered[['profile_cluster']].reset_index()
cluster_map.columns = [unique_id, 'profile_cluster']

df_clustered = df.merge(cluster_map, on=unique_id, how='left')

# Drop the 'profile' column (replaced by profile_cluster)
if 'profile' in df_clustered.columns:
    df_clustered.drop(columns=['profile'], inplace=True)

print(f"Final clustered dataframe: {df_clustered.shape}")
print(f"\nProfile-cluster distribution (rows):")
print(df_clustered['profile_cluster'].value_counts().sort_index())

# Save to Lakehouse
df_clustered_spark = spark.createDataFrame(df_clustered)
df_clustered_spark.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(OUTPUT_TABLE)
print(f"\nâœ… Saved to Lakehouse: {OUTPUT_TABLE}")
print(f"   Rows: {len(df_clustered)}, Columns: {len(df_clustered.columns)}")

## Summary

| Cluster | Stores | Archetypes | Avg $/wk | Christmas Lift |
|---------|--------|-----------|----------|----------------|
| regular_0 | 5 | 4M, 1S | $407K | +27.1% |
| regular_1 | 15 | 11M, 4L | $749K | +22.2% |
| regular_2 | 4 | 2L, 1M, 1S | $487K | +25.6% |
| regular_3 | 14 | 7M, 5S, 2L | $507K | +25.1% |
| erratic | 12 | 7S, 5M | $323K | â€” |

**Method:** K-Means on z-score normalized weekly sales curves (shape-based)  
**Output table:** `supermarket_net_sales_forecast_clustered` (5,200 rows Ã— 100 columns)